# 11 - Credit bureau features (v2 experiment)

Tests whether credit bureau data improves the scorecard. Each client's loans at other lenders (bureau.csv) are aggregated into a small set of features with clear business meaning, screened with WoE/IV, and the scorecard is refitted on the same train/test split. The ECL, staging and stress test are not rebuilt; the result documents whether bureau features would be worth adopting in a future model version.

In [1]:
   %load_ext autoreload
   %autoreload 2

In [2]:
import pandas as pd

bureau = pd.read_csv("../data/raw/bureau.csv")
train = pd.read_parquet("../data/processed/train.parquet")

print(f"Rows: {len(bureau):,}  Columns: {bureau.shape[1]}")
print(f"Clients with bureau records: {bureau['SK_ID_CURR'].nunique():,}")

loans_per_client = bureau.groupby("SK_ID_CURR").size()
print(f"Loans per client: median {loans_per_client.median():.0f}, max {loans_per_client.max()}")

coverage = train["SK_ID_CURR"].isin(bureau["SK_ID_CURR"]).mean()
print(f"Share of training clients with bureau history: {coverage:.1%}\n")

print(bureau["CREDIT_ACTIVE"].value_counts(), "\n")
print(bureau["CREDIT_TYPE"].value_counts().head(), "\n")
bureau.head().T

Rows: 1,716,428  Columns: 17
Clients with bureau records: 305,811
Loans per client: median 4, max 116
Share of training clients with bureau history: 85.7%

CREDIT_ACTIVE
Closed      1079273
Active       630607
Sold           6527
Bad debt         21
Name: count, dtype: int64 

CREDIT_TYPE
Consumer credit    1251615
Credit card         402195
Car loan             27690
Mortgage             18391
Microloan            12413
Name: count, dtype: int64 



,0,1,2,3,4
SK_ID_CURR,215354,215354,215354,215354,215354
SK_ID_BUREAU,5714462,5714463,5714464,5714465,5714466
CREDIT_ACTIVE,Closed,Active,Active,Active,Active
CREDIT_CURRENCY,currency 1,currency 1,currency 1,currency 1,currency 1
DAYS_CREDIT,-497,-208,-203,-203,-629
CREDIT_DAY_OVERDUE,0,0,0,0,0
DAYS_CREDIT_ENDDATE,-153.0,1075.0,528.0,NaN,1197.0
DAYS_ENDDATE_FACT,-153.0,NaN,NaN,NaN,NaN
AMT_CREDIT_MAX_OVERDUE,NaN,NaN,NaN,NaN,77674.5
CNT_CREDIT_PROLONG,0,0,0,0,0


In [3]:
import sys
sys.path.append("..")
from src.bureau import aggregate_bureau

bureau_agg = aggregate_bureau(bureau)
print(bureau_agg.shape)
bureau_agg.describe().T

(305811, 10)


,count,mean,std,min,25%,50%,75%,max
SK_ID_CURR,305811.0,278047.300091,1.028496e+05,100001.00,188878.5,277895.000,367184.5,4.562550e+05
BUREAU_LOAN_COUNT,305811.0,5.612709,4.430354e+00,1.00,2.0,4.000,8.0,1.160000e+02
BUREAU_ACTIVE_COUNT,305811.0,2.062081,1.791724e+00,0.00,1.0,2.000,3.0,3.200000e+01
BUREAU_ACTIVE_DEBT,305811.0,643479.872358,1.622062e+06,-6978830.49,0.0,167641.515,664438.5,3.344983e+08
BUREAU_MAX_DAYS_OVERDUE,305811.0,4.436629,8.557359e+01,0.00,0.0,0.000,0.0,2.792000e+03
BUREAU_OVERDUE_AMOUNT,305811.0,212.793258,1.569161e+04,0.00,0.0,0.000,0.0,3.756681e+06
BUREAU_DAYS_SINCE_LAST_CREDIT,305811.0,-490.942608,5.335293e+02,-2922.00,-623.0,-305.000,-149.0,0.000000e+00
BUREAU_SOLD_OR_BAD_COUNT,305811.0,0.021412,1.586357e-01,0.00,0.0,0.000,0.0,9.000000e+00
BUREAU_MICROLOAN_COUNT,305811.0,0.040590,5.453237e-01,0.00,0.0,0.000,0.0,7.900000e+01
BUREAU_ANY_DELINQUENCY,305811.0,0.027115,1.624182e-01,0.00,0.0,0.000,0.0,1.000000e+00


In [4]:
print((bureau_agg["BUREAU_ACTIVE_DEBT"] < 0).sum())

1091


### Findings: 

The negative debt: 1,091 clients, about 0.4%. That’s small enough  and we leave as it is.

## Join to the training set and calculate IV

In [5]:
import numpy as np
from optbinning import OptimalBinning

train_b = train.merge(bureau_agg, on="SK_ID_CURR", how="left")
BUREAU_FEATURES = [c for c in bureau_agg.columns if c != "SK_ID_CURR"]

rows, bureau_binners = [], {}
for var in BUREAU_FEATURES:
    optb = OptimalBinning(name=var, dtype="numerical", monotonic_trend="auto",
                          max_n_bins=8, min_bin_size=0.05)
    optb.fit(train_b[var], train_b["TARGET"])
    optb.binning_table.build()
    rows.append({"variable": var, "iv": optb.binning_table.iv})
    bureau_binners[var] = optb

bureau_iv = pd.DataFrame(rows).sort_values("iv", ascending=False)
print(bureau_iv.round(3))

                        variable     iv
5  BUREAU_DAYS_SINCE_LAST_CREDIT  0.089
1            BUREAU_ACTIVE_COUNT  0.058
2             BUREAU_ACTIVE_DEBT  0.049
0              BUREAU_LOAN_COUNT  0.016
3        BUREAU_MAX_DAYS_OVERDUE  0.013
4          BUREAU_OVERDUE_AMOUNT  0.013
6       BUREAU_SOLD_OR_BAD_COUNT  0.013
7         BUREAU_MICROLOAN_COUNT  0.013
8         BUREAU_ANY_DELINQUENCY  0.013


In [6]:
for var in ["BUREAU_MAX_DAYS_OVERDUE", "BUREAU_OVERDUE_AMOUNT",
            "BUREAU_SOLD_OR_BAD_COUNT", "BUREAU_MICROLOAN_COUNT"]:
    share = (train_b[var] > 0).mean()
    rate = train_b.loc[train_b[var] > 0, "TARGET"].mean()
    print(f"{var}: {share:.1%} of clients > 0, default rate {rate:.1%}")

BUREAU_MAX_DAYS_OVERDUE: 1.1% of clients > 0, default rate 15.7%
BUREAU_OVERDUE_AMOUNT: 1.1% of clients > 0, default rate 16.0%
BUREAU_SOLD_OR_BAD_COUNT: 1.7% of clients > 0, default rate 10.2%
BUREAU_MICROLOAN_COUNT: 1.1% of clients > 0, default rate 20.4%


### Rare signals and a smaller minimum bin size

Four sparse features (MAX_DAYS_OVERDUE, OVERDUE_AMOUNT, SOLD_OR_BAD_COUNT, MICROLOAN_COUNT) had identical IVs of 0.013. Fewer than 5% of clients have non-zero values, so with min_bin_size = 0.05 the binning could not create a separate bin for them; their IV came only from the Missing bin (clients without bureau history).

The rare groups carry strong signals: clients with microloans default at 20.4% and those with current overdue loans at about 16%, versus 8.07% on average. Each group holds about 1.1% of clients, around 2,400 in the training set with 400–500 defaults, enough for a reliable bin.

Two changes:
- **BUREAU_ANY_DELINQUENCY:** a combined flag for any current overdue loan or microloan use. One larger, more stable group is easier to explain than several sparse features. Sold or written-off loans are left out, since their signal is weak (10.2%).
- **min_bin_size = 0.01** for the bureau features, so small but strong groups can form their own bin. Smaller bins are less stable, which is accepted here because the groups contain enough defaults and the signals have a clear business meaning.

In [7]:
bureau_agg = aggregate_bureau(bureau)
train_b = train.merge(bureau_agg, on="SK_ID_CURR", how="left")
BUREAU_FEATURES = [c for c in bureau_agg.columns if c != "SK_ID_CURR"]

rows, bureau_binners = [], {}
for var in BUREAU_FEATURES:
    optb = OptimalBinning(name=var, dtype="numerical", monotonic_trend="auto",
                          max_n_bins=8, min_bin_size=0.01, min_prebin_size=0.01)
    optb.fit(train_b[var], train_b["TARGET"])
    optb.binning_table.build()
    rows.append({"variable": var, "iv": optb.binning_table.iv})
    bureau_binners[var] = optb

bureau_iv = pd.DataFrame(rows).sort_values("iv", ascending=False)
print(bureau_iv.round(3))

                        variable     iv
5  BUREAU_DAYS_SINCE_LAST_CREDIT  0.091
1            BUREAU_ACTIVE_COUNT  0.061
2             BUREAU_ACTIVE_DEBT  0.054
8         BUREAU_ANY_DELINQUENCY  0.044
7         BUREAU_MICROLOAN_COUNT  0.036
4          BUREAU_OVERDUE_AMOUNT  0.024
3        BUREAU_MAX_DAYS_OVERDUE  0.023
0              BUREAU_LOAN_COUNT  0.016
6       BUREAU_SOLD_OR_BAD_COUNT  0.015


### Check correlations, refit, and compare the Gini

In [8]:
import joblib
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score
from src.woe import load_binners, woe_transform

binners = load_binners("../models/binners.joblib")
FINAL_VARS = joblib.load("../models/pd_model.joblib")["final_vars"]
BUREAU_CANDIDATES = ["BUREAU_DAYS_SINCE_LAST_CREDIT", "BUREAU_ACTIVE_COUNT",
                     "BUREAU_ACTIVE_DEBT", "BUREAU_ANY_DELINQUENCY"]
bureau_sel = {v: bureau_binners[v] for v in BUREAU_CANDIDATES}

test = pd.read_parquet("../data/processed/test.parquet")
test_b = test.merge(bureau_agg, on="SK_ID_CURR", how="left")

def woe_all(df):
    return pd.concat([woe_transform(df, binners)[FINAL_VARS],
                      woe_transform(df, bureau_sel)], axis=1)

X_train, X_test = woe_all(train_b), woe_all(test_b)
y_train, y_test = train_b["TARGET"], test_b["TARGET"]

# Overlap between the new features and all others
corr = X_train.corr().abs()
for var in BUREAU_CANDIDATES:
    top = corr[var].drop(var).sort_values(ascending=False).head(2)
    print(f"{var}: " + ", ".join(f"{k} {v:.2f}" for k, v in top.items()))

def gini_scores(cols):
    model = LogisticRegression(C=np.inf, max_iter=1000)
    cv = cross_val_score(model, X_train[cols], y_train, cv=5, scoring="roc_auc").mean()
    model.fit(X_train[cols], y_train)
    test_auc = roc_auc_score(y_test, model.predict_proba(X_test[cols])[:, 1])
    return 2 * cv - 1, 2 * test_auc - 1, model

for name, cols in [("Scorecard (12)", FINAL_VARS),
                   ("Scorecard + bureau (16)", FINAL_VARS + BUREAU_CANDIDATES)]:
    cv_g, test_g, model = gini_scores(cols)
    print(f"\n{name}: CV Gini {cv_g:.4f}, test Gini {test_g:.4f}")

coefs = pd.Series(model.coef_[0], index=FINAL_VARS + BUREAU_CANDIDATES)
print("\n", coefs[BUREAU_CANDIDATES].round(3))

BUREAU_DAYS_SINCE_LAST_CREDIT: BUREAU_ACTIVE_COUNT 0.44, BUREAU_ACTIVE_DEBT 0.40
BUREAU_ACTIVE_COUNT: BUREAU_ACTIVE_DEBT 0.58, BUREAU_DAYS_SINCE_LAST_CREDIT 0.44
BUREAU_ACTIVE_DEBT: BUREAU_ACTIVE_COUNT 0.58, BUREAU_DAYS_SINCE_LAST_CREDIT 0.40
BUREAU_ANY_DELINQUENCY: BUREAU_DAYS_SINCE_LAST_CREDIT 0.16, EXT_SOURCE_3 0.16

Scorecard (12): CV Gini 0.4912, test Gini 0.4897

Scorecard + bureau (16): CV Gini 0.4949, test Gini 0.4919

 BUREAU_DAYS_SINCE_LAST_CREDIT   -0.138
BUREAU_ACTIVE_COUNT             -0.310
BUREAU_ACTIVE_DEBT              -0.143
BUREAU_ANY_DELINQUENCY          -0.424
dtype: float64


## Findings

**IV screening:** four bureau features pass IV screening with clear business meaning: recency of the last loan (0.091), number of active loans (0.061), active debt (0.054) and a delinquency flag for current overdue loans or microloan use (0.044). Rare signals required smaller bins (min_bin_size and min_prebin_size = 0.01): clients with microloans default at 20.4% and those with current overdue loans at about 16%, but each group holds only about 1% of clients.

**Overlap:** no WoE correlation exceeds 0.6. The bureau features overlap mainly with each other (active count and active debt: 0.58), and only weakly with the application variables.

**Model comparison (same train/test split):**

| Model | CV Gini | Test Gini |
|---|---|---|
| Scorecard (12 variables) | 0.4912 | 0.4897 |
| Scorecard + bureau (16) | 0.4949 | 0.4919 |

All four bureau coefficients are negative, as expected, but the gain is small (+0.002 to +0.004 Gini). The external scores (EXT_SOURCE_1/2/3) most likely already contain credit bureau information, so the bureau features add little beyond them.

**Decision:** bureau features are not adopted in the current model version. The small gain does not justify four additional variables, a second data source and a more complex pipeline. BUREAU_ANY_DELINQUENCY is the strongest candidate for a future version: simple, intuitive and nearly independent of the existing variables. The bureau data also does not close the gap to the XGBoost challenger (Gini 0.531), whose advantage comes from interactions rather than missing data.